# Online BCI Decisions — Buffer, Confidence, and Asynchronous Control

*Assumes the material of notebook #6 (online simulation, causal filtering, ITR, latency-accuracy tradeoff).*

Notebook #6 built the online loop and showed that offline accuracy overestimates real-time performance. This notebook asks the practical questions that arise when deploying that loop: **How large should the buffer be? When should the system commit to a decision? What if the user is not performing imagery at all?**

These are not parameter-tuning details — they determine whether the BCI is usable or frustrating. Each scenario uses the same EEGBCI simulation infrastructure from notebook #6.

## Table of contents

1. **Setup** — Reproduce the trained classifier and streaming infrastructure.
2. **Scenario A: The buffer engineer** — *"How large should my buffer be?"* → The memory-latency-accuracy triangle.
3. **Scenario B: The confidence designer** — *"When should the system commit to a decision?"* → Thresholding, evidence accumulation, and the cost of silence.
4. **Scenario C: The asynchronous challenge** — *"What if the user is not doing anything?"* → Idle-state detection and the false-positive problem.
5. **Scenario D: The P300 averaging strategist** — *"How many stimulus repetitions before I decide?"* → The speed-accuracy tradeoff in ERP-BCIs.
6. **Synthesis** — An online BCI design checklist.
7. **Practice scenarios**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfilt, sosfilt_zi

import mne
from mne.datasets import eegbci, sample as mne_sample
from mne.channels import make_standard_montage
from mne.io import concatenate_raws, read_raw_edf
from mne.decoding import CSP

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

In [ ]:
# ── Reproduce the trained MI classifier from notebook #6. ──
def load_mi_raw(subject, runs):
    raw_fnames = eegbci.load_data(subject, runs, update_path=True)
    raw = concatenate_raws([read_raw_edf(f, preload=True) for f in raw_fnames])
    eegbci.standardize(raw)
    raw.set_montage(make_standard_montage("standard_1005"))
    raw.filter(l_freq=7.0, h_freq=30.0)
    return raw

raw_train = load_mi_raw(1, [4, 8])
events_train, eid_train = mne.events_from_annotations(raw_train)
eid_lr = {k: v for k, v in eid_train.items() if v in (2, 3)}
epochs_train = mne.Epochs(raw_train, events_train, eid_lr, tmin=0.5, tmax=3.5,
                          baseline=None, picks="eeg", preload=True)
X_train = epochs_train.get_data(copy=False) * 1e6
y_train = epochs_train.events[:, -1]

clf = Pipeline([("CSP", CSP(n_components=4, reg=None, log=True, norm_trace=False)),
                ("LDA", LinearDiscriminantAnalysis())])
clf.fit(X_train, y_train)

# Load the test run.
raw_test = load_mi_raw(1, [12])
events_test, eid_test = mne.events_from_annotations(raw_test)
sfreq = raw_test.info["sfreq"]
data_test = raw_test.get_data(picks="eeg") * 1e6
n_channels, n_samples = data_test.shape
true_events = [(ev[0], ev[2]) for ev in events_test if ev[2] in (2, 3)]

def itr_bits_per_min(n_classes, accuracy, trial_duration_s):
    P = np.clip(accuracy, 1e-10, 1 - 1e-10)
    N = n_classes
    bits = np.log2(N) + P * np.log2(P) + (1 - P) * np.log2((1 - P) / (N - 1))
    return (60.0 / trial_duration_s) * bits

print(f"Classifier trained. Test stream: {n_samples/sfreq:.0f} s, {len(true_events)} events.")

---

## 2. Scenario A — The buffer engineer

### The situation

> *"I'm deploying on an embedded device with limited RAM. How large does the data buffer need to be? What do I lose if I shrink it?"*

### ❓ Pause — your prediction

1. The buffer stores the most recent N seconds of EEG. If the classifier needs a 2-second window, can the buffer be exactly 2 seconds? Or does it need to be longer?
2. What happens if the buffer is *longer* than needed — say, 10 seconds? Any downside?
3. The causal filter has a transient ("ringing") at the start. How does buffer length interact with filter startup?

---

In [ ]:
# Demonstrate the filter startup problem.
# Apply a causal filter to progressively shorter buffers and compare the output.
sos = butter(5, [7, 30], btype="band", fs=sfreq, output="sos")

# Take a 5-second chunk of data from channel 0.
chunk = data_test[0, :int(5 * sfreq)]

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True, sharey=True)
buffer_lengths = [0.5, 1.0, 2.0, 5.0]

for ax, buf_sec in zip(axes, buffer_lengths):
    buf_samp = int(buf_sec * sfreq)
    # Simulate: we only have the last buf_samp samples.
    short_chunk = chunk[-buf_samp:]
    zi = sosfilt_zi(sos) * short_chunk[0]
    filtered, _ = sosfilt(sos, short_chunk, zi=zi)

    t = np.arange(len(filtered)) / sfreq
    ax.plot(t, filtered, color="steelblue", linewidth=0.8)
    # Mark the startup transient region (approximately first 0.3 s).
    ax.axvspan(0, 0.3, alpha=0.15, color="red")
    ax.set_ylabel("µV")
    ax.set_title(f"Buffer = {buf_sec} s", fontsize=10, loc="left")

axes[-1].set_xlabel("Time within buffer (s)")
plt.suptitle("Filter startup transient vs buffer length", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### Interpretation

The red-shaded region marks the filter's **startup transient** — the first ~0.3 seconds where the filter output is unreliable because the filter state has not yet settled. In the 0.5 s buffer, this transient occupies 60% of the data — the feature extraction window is severely corrupted. In the 5 s buffer, the transient is negligible.

**The practical rule:** The buffer should be at least `feature_window + 0.5 s` to allow the filter transient to decay before the feature window begins. For a 2 s MI window, a minimum 2.5 s buffer; for safety, 3 s.

A better approach — used in production systems — is to maintain a **persistent filter state**: instead of re-initialising the filter for each buffer, the filter state carries over from the previous update. This eliminates the startup transient entirely, at the cost of requiring the filter to run continuously.

---

❓ **Exercise.** Modify the simulation loop from notebook #6 to maintain a persistent `sosfilt` state across updates (using the `zi` output of each call as the `zi` input of the next). Compare the classification accuracy to the re-initialised version.

---

## 3. Scenario B — The confidence designer

### The situation

> *"The classifier emits a prediction every 100 ms, but many predictions are wrong — especially during transitions. Can I make the system wait until it's confident before outputting a decision?"*

### ❓ Pause — your prediction

1. If you threshold on classifier confidence, what happens to accuracy? To latency? To ITR?
2. What should the system output when it is *not* confident — silence, or the previous decision?
3. Is there a risk of the system *never* becoming confident for some trials?

---

In [ ]:
# Run the streaming simulation with confidence thresholding.
window_samp = int(2.0 * sfreq)
step_samp = int(0.1 * sfreq)

# Compute the full prediction stream.
all_times, all_probs = [], []
for start in range(0, n_samples - window_samp, step_samp):
    end = start + window_samp
    X_w = data_test[:, start:end][np.newaxis, :, :]
    prob = clf.decision_function(X_w)[0]
    all_times.append(end)
    all_probs.append(prob)
all_times = np.array(all_times)
all_probs = np.array(all_probs)

# Evaluate at different confidence thresholds.
thresholds = [0.0, 0.3, 0.6, 1.0, 1.5]

print(f"{'Threshold':<12s} {'Accuracy':>10s} {'Decided':>10s} {'Undecided':>10s} {'Mean latency':>14s}")
print("-" * 60)

threshold_results = []

for thr in thresholds:
    correct, total, undecided_count = 0, 0, 0
    latencies = []

    for ev_sample, ev_label in true_events:
        t_start = ev_sample + int(0.5 * sfreq)
        t_end   = ev_sample + int(3.5 * sfreq)
        mask = (all_times >= t_start) & (all_times <= t_end)
        if not np.any(mask):
            continue

        probs_trial = all_probs[mask]
        times_trial = all_times[mask]

        # Find the first prediction that exceeds the threshold.
        confident = np.abs(probs_trial) >= thr
        if not np.any(confident):
            undecided_count += 1
            total += 1
            continue

        first_idx = np.argmax(confident)
        pred = 3 if probs_trial[first_idx] > 0 else 2
        correct += int(pred == ev_label)
        total += 1
        latency = (times_trial[first_idx] - ev_sample) / sfreq
        latencies.append(latency)

    acc = correct / (total - undecided_count) if (total - undecided_count) > 0 else 0
    mlat = np.mean(latencies) if latencies else float('nan')
    threshold_results.append((thr, acc, total - undecided_count, undecided_count, mlat))
    print(f"{thr:<12.1f} {acc:>9.1%} {total-undecided_count:>10d} {undecided_count:>10d} {mlat:>12.2f} s")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

thrs = [r[0] for r in threshold_results]
accs = [r[1] for r in threshold_results]
lats = [r[4] for r in threshold_results]
undec = [r[3] for r in threshold_results]

ax1.plot(thrs, accs, "o-", color="#2980b9", linewidth=2, label="Accuracy")
ax1b = ax1.twinx()
ax1b.plot(thrs, undec, "s--", color="#e74c3c", linewidth=1.5, label="Undecided trials")
ax1.set_xlabel("Confidence threshold")
ax1.set_ylabel("Accuracy (among decided trials)", color="#2980b9")
ax1b.set_ylabel("Undecided trials", color="#e74c3c")
ax1.set_title("Higher threshold → better accuracy, more rejected trials")

ax2.plot(thrs, lats, "o-", color="#27ae60", linewidth=2)
ax2.set_xlabel("Confidence threshold")
ax2.set_ylabel("Mean latency to decision (s)")
ax2.set_title("Higher threshold → longer wait")

plt.tight_layout()
plt.show()

### Interpretation

The confidence threshold creates a three-way tradeoff:

| Threshold | Accuracy | Latency | Undecided trials |
|---|---|---|---|
| Low (0.0) | Moderate | Short | None |
| Medium | Higher | Longer | Some |
| High (1.5) | Very high | Very long | Many |

A high threshold makes the system more accurate *when it decides*, but it refuses to decide on difficult trials. In a real BCI, an undecided trial is not free — the user has spent effort on imagery for no output. The optimal threshold depends on the cost structure: Is a wrong decision worse than no decision? For a wheelchair controller, yes. For a game, probably not.

---

## 4. Scenario C — The asynchronous challenge

### The situation

> *"In our demo, the system was cue-based — the user was told when to imagine. But in real use, the user decides when to issue a command. How do I tell the difference between 'imagining left hand' and 'not doing anything'?"*

This is the **synchronous vs asynchronous** distinction — the single biggest gap between laboratory BCIs and real-world use.

### ❓ Pause — your prediction

1. A 2-class MI classifier (left vs right) always outputs a prediction. During rest, what does it output?
2. How would you add a third class ("idle" / "no command") without collecting additional training data?
3. What is the consequence of a false positive (the system thinks the user issued a command when they are resting)?

---

In [ ]:
# Analyse the classifier's behaviour during REST periods.
# Rest = the intervals between trials (not within the 0.5–3.5 s imagery window).

# Create masks for imagery and rest periods.
is_imagery = np.zeros(len(all_times), dtype=bool)
is_rest = np.ones(len(all_times), dtype=bool)

for ev_sample, ev_label in true_events:
    t_start = ev_sample + int(0.5 * sfreq)
    t_end   = ev_sample + int(3.5 * sfreq)
    mask = (all_times >= t_start) & (all_times <= t_end)
    is_imagery |= mask
    # Also exclude a wider window around events.
    mask_wide = (all_times >= ev_sample - int(1 * sfreq)) & (all_times <= ev_sample + int(5 * sfreq))
    is_rest &= ~mask_wide

probs_imagery = all_probs[is_imagery]
probs_rest = all_probs[is_rest]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

ax1.hist(probs_imagery, bins=40, color="#2980b9", alpha=0.7, edgecolor="white")
ax1.set_xlabel("Classifier confidence")
ax1.set_ylabel("Count")
ax1.set_title(f"During IMAGERY ({np.sum(is_imagery)} windows)")
ax1.axvline(0, color="grey", linestyle="--")

ax2.hist(probs_rest, bins=40, color="#e74c3c", alpha=0.7, edgecolor="white")
ax2.set_xlabel("Classifier confidence")
ax2.set_title(f"During REST ({np.sum(is_rest)} windows)")
ax2.axvline(0, color="grey", linestyle="--")

plt.suptitle("Classifier output distribution: imagery vs rest", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"Imagery: mean |confidence| = {np.abs(probs_imagery).mean():.2f}")
print(f"Rest:    mean |confidence| = {np.abs(probs_rest).mean():.2f}")

### Interpretation

During rest, the classifier output should be centred near zero with moderate spread — essentially random. During imagery, the distribution should be shifted towards the correct class (bimodal: negative for left, positive for right).

If the distributions overlap substantially, a simple confidence threshold cannot reliably distinguish "imagery" from "rest." In that case, the system will produce **false positives** during rest — outputting unwanted commands when the user is idle.

### Approaches to asynchronous control

| Approach | Mechanism | Pros | Cons |
|---|---|---|---|
| **Confidence threshold** | Suppress output when \|d\| < threshold | Simple | Misses weak imagery; cannot eliminate all false positives |
| **Idle-state classifier** | Train a 3-class model (left, right, rest) | Explicitly models rest | Requires rest-state training data; harder to train |
| **Evidence accumulation** | Require N consecutive consistent predictions | Reduces transient errors | Adds latency |
| **User-initiated trigger** | Physical switch to enable BCI mode | Eliminates false positives | Requires residual motor ability |

No single approach solves the problem completely. Production BCIs typically combine confidence thresholding with evidence accumulation.

---

❓ **Exercise.** Implement a simple evidence accumulation rule: the system only outputs a command when 5 consecutive predictions agree. How does this affect the false-positive rate during rest, and the true-positive latency during imagery?

---

## 5. Scenario D — The P300 averaging strategist

### The situation

> *"In a P300 speller, each row/column flashes multiple times. More repetitions give a cleaner ERP and better accuracy — but the user has to wait longer. How many repetitions should I use?"*

This is the ERP-specific version of the latency-accuracy tradeoff. Instead of varying window length, we vary the **number of stimulus repetitions** averaged before classification.

In [ ]:
# Simulate P300-like averaging with the MNE sample dataset.
# We use auditory trials and measure how classification improves with averaging.

from mne.decoding import Vectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

data_path = mne_sample.data_path()
raw_p3 = mne.io.read_raw_fif(data_path / "MEG" / "sample" / "sample_audvis_raw.fif",
                               preload=True)
events_p3 = mne.find_events(raw_p3, stim_channel="STI 014")
raw_p3.pick("eeg")
raw_p3.filter(0.1, 40.0, method="iir",
              iir_params=dict(order=5, ftype="butter"), phase="minimum")
raw_p3.set_eeg_reference("average", projection=True)
raw_p3.apply_proj()

# Relabel events.
events_p3_av = events_p3[np.isin(events_p3[:, -1], [1, 2, 3, 4])].copy()
events_p3_av[np.isin(events_p3_av[:, -1], [1, 2]), -1] = 0  # auditory
events_p3_av[np.isin(events_p3_av[:, -1], [3, 4]), -1] = 1  # visual

epochs_p3 = mne.Epochs(raw_p3, events_p3_av, {"aud": 0, "vis": 1},
                        tmin=0.0, tmax=0.5, baseline=None, picks="eeg",
                        preload=True, reject=dict(eeg=100e-6))

X_p3 = epochs_p3.get_data(copy=False)
y_p3 = epochs_p3.events[:, -1]

# Train on first half.
split = len(y_p3) // 2
clf_p3 = make_pipeline(Vectorizer(), StandardScaler(),
                        LogisticRegression(solver="liblinear"))
clf_p3.fit(X_p3[:split], y_p3[:split])

# Test on second half — simulate averaging N trials before classifying.
X_test_p3 = X_p3[split:]
y_test_p3 = y_p3[split:]

print(f"Test epochs: {len(y_test_p3)} (aud: {np.sum(y_test_p3==0)}, vis: {np.sum(y_test_p3==1)})")

In [ ]:
# Simulate: average N random epochs of the same class, then classify.
n_reps_list = [1, 2, 3, 5, 8, 12, 20]
n_simulations = 200  # random draws per condition

rng = np.random.RandomState(42)
avg_results = []

for n_reps in n_reps_list:
    corrects = 0
    totals = 0
    for _ in range(n_simulations):
        # Pick a random class.
        true_class = rng.choice([0, 1])
        class_idx = np.where(y_test_p3 == true_class)[0]
        if len(class_idx) < n_reps:
            continue
        # Draw N random trials and average them.
        chosen = rng.choice(class_idx, size=n_reps, replace=False)
        averaged = X_test_p3[chosen].mean(axis=0, keepdims=True)
        pred = clf_p3.predict(averaged)[0]
        corrects += int(pred == true_class)
        totals += 1

    acc = corrects / totals
    # Simulate timing: each repetition takes ~0.5 s of stimulus + ISI.
    trial_time = n_reps * 0.8  # 0.8 s per repetition (stimulus + ISI)
    rate = itr_bits_per_min(2, acc, trial_time)
    avg_results.append((n_reps, acc, trial_time, rate))
    print(f"{n_reps:2d} repetitions: accuracy={acc:.1%}, time={trial_time:.1f}s, ITR={rate:.1f} b/m")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

nreps = [r[0] for r in avg_results]
accs = [r[1] for r in avg_results]
itrs = [r[3] for r in avg_results]

ax1.plot(nreps, accs, "o-", color="#2980b9", linewidth=2)
ax1.set_xlabel("Number of repetitions")
ax1.set_ylabel("Accuracy")
ax1.set_title("More averaging → higher accuracy")
ax1.axhline(0.5, color="grey", linestyle="--")

ax2.plot(nreps, itrs, "o-", color="#e74c3c", linewidth=2)
ax2.set_xlabel("Number of repetitions")
ax2.set_ylabel("ITR (bits/min)")
ax2.set_title("Optimal repetition count maximises ITR")

plt.suptitle("P300-like averaging tradeoff", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

best_idx = np.argmax(itrs)
print(f"Optimal: {nreps[best_idx]} repetitions "
      f"(accuracy={accs[best_idx]:.1%}, ITR={itrs[best_idx]:.1f} bits/min)")

### Interpretation

Averaging follows the law of large numbers: each additional repetition reduces the noise variance by 1/N. But ITR peaks at a small number of repetitions — typically 3–5 — because the marginal accuracy gain of each additional repetition shrinks while the time cost is constant.

In a real P300 speller, the optimal repetition count is user-dependent. Experienced users with strong P300 signals may need only 2–3 repetitions; novice users or those with neurological conditions may need 10+. Adaptive systems dynamically adjust the number of repetitions based on real-time confidence estimation.

---

❓ **Exercise.** Implement an **adaptive stopping** rule: after each repetition, check the classifier's confidence. If it exceeds a threshold, stop averaging and output the decision immediately. If not, continue averaging. Compare the mean latency and accuracy to the fixed-repetition approach.

---

## 6. Synthesis — an online BCI design checklist

### Before deploying any BCI online, answer these questions:

**1. Filtering strategy** (Notebook #6)
- Use causal filters. Train the classifier on causal-filtered data.
- Maintain persistent filter state across updates (no re-initialisation).

**2. Buffer sizing** (Scenario A)
- Buffer ≥ feature window + 0.5 s for filter settling.
- Prefer persistent filter state to eliminate the startup transient.

**3. Decision policy** (Scenario B)
- Choose a confidence threshold based on the cost of errors vs the cost of silence.
- Consider evidence accumulation (N consecutive agreements) for noisy signals.

**4. Asynchronous handling** (Scenario C)
- A 2-class classifier produces random output during rest. This *must* be addressed.
- Options: confidence threshold, idle-state classifier, or user-initiated trigger.

**5. Repetition/averaging strategy** (Scenario D, ERP systems only)
- Fixed repetitions: simple, predictable latency.
- Adaptive stopping: variable latency, potentially higher ITR.
- The optimal count is user-dependent; adapt over sessions.

**6. Evaluation** (Always)
- Report ITR, not just accuracy.
- Report false-positive rate during rest (for asynchronous systems).
- Simulate before deploying — the simulation loop from notebook #6 reveals problems that offline cross-validation hides.

---

## 7. Practice scenarios

### Scenario P1

> A BCI-controlled wheelchair must detect "go forward" (right-hand imagery) vs "stop" (rest — no imagery). The cost of a false "go forward" during rest is a collision. What design choices minimise this risk?

<details>
<summary>Click to reveal the analysis</summary>

- **Asymmetric cost:** A false positive ("go" during rest) is dangerous; a false negative ("stop" during intended "go") is merely inconvenient. The confidence threshold should be high, biased towards the safe default (stop).
- **Evidence accumulation:** Require 5+ consecutive "go" predictions before the wheelchair moves. This adds ~0.5 s latency but dramatically reduces false starts.
- **Timeout:** After issuing "go," the system should automatically revert to "stop" if confidence drops below threshold for >1 second.
- **Physical override:** A hardware stop button is not optional — it is a safety requirement.
</details>

### Scenario P2

> A P300 speller user has a weak P300 signal. With 5 repetitions, accuracy is 70% (above chance but low). The clinician asks: should I increase to 15 repetitions or switch to a different paradigm?

<details>
<summary>Click to reveal the analysis</summary>

- **Compute the ITR.** At 70% with 5 reps (4 s total): ITR ≈ 2.4 bits/min for a 36-character matrix. At 90% with 15 reps (12 s): ITR ≈ 2.8 bits/min. The gain from tripling the repetitions is modest.
- **Consider the user experience.** 12 seconds per character means ~5 characters per minute at best. This is extremely slow for communication.
- **Alternative paradigm.** If the user has residual eye movement, an SSVEP-based system may achieve higher ITR. If they lack reliable visual fixation, an auditory P300 paradigm (with sounds instead of flashes) may produce a stronger ERP.
- **The decision is clinical, not just engineering.** Accuracy numbers must be translated into *functional communication rate* for the clinical team.
</details>

### Scenario P3

> Your online BCI demo works perfectly in the lab, but when you demonstrate it at a conference, accuracy drops from 80% to 55%. The user is the same person. What went wrong?

<details>
<summary>Click to reveal the analysis</summary>

Several factors change between lab and conference:
- **Electrical noise.** Conference halls have different mains noise, Wi-Fi interference, and grounding. Check the PSD for unexpected peaks.
- **Electrode impedance.** If setup was rushed, impedances may be high, increasing noise.
- **Cognitive load.** The user is distracted by the audience, conversations, and stress. Motor imagery requires concentration — divided attention degrades the signal.
- **Muscle artifacts.** The user may be tenser, producing more EMG contamination in the beta band — exactly where the MI classifier looks.
- **The lesson:** Online BCI performance is not a fixed number. It depends on the environment, the user's state, and the setup quality. Always test in the deployment environment.
</details>

---

## 8. Key takeaways

1. **Buffer sizing is not trivial.** The buffer must accommodate the feature window plus the filter's settling time. Persistent filter state eliminates startup transients.

2. **Confidence thresholding trades accuracy for coverage.** Higher thresholds increase accuracy among decided trials but reject difficult trials entirely. The optimal threshold depends on the cost of errors vs silence.

3. **Asynchronous operation is the hardest problem in BCI.** A 2-class classifier produces random output during rest. Every deployed system must handle the idle state explicitly.

4. **Averaging repetitions follow diminishing returns.** For ERP-BCIs, the ITR-optimal number of repetitions is typically small (3–5). Adaptive stopping can improve ITR further.

5. **Lab performance ≠ field performance.** Environment, setup quality, and user state all affect online accuracy. Test where you deploy.